# Reto 08 — SOLUCION

No liberar hasta cerrar el modulo.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

ventas = pd.read_csv("../../../_RECURSOS/datasets/ventas.csv",
                     sep=";", decimal=",", encoding="utf-8")

## 0 — Preparar

In [ ]:
ventas = ventas.drop_duplicates().reset_index(drop=True)
ventas["canal"] = ventas["canal"].str.strip().str.title()
ventas["fecha"] = pd.to_datetime(ventas["fecha"], format="mixed", dayfirst=True)
ventas["importe"] = ventas["importe"].fillna(
    (ventas["cantidad"] * ventas["precio_unitario"]
     * (1 - ventas["descuento_pct"] / 100)).round(2)
)
ventas = ventas[ventas["cantidad"] > 0].copy()
ventas["mes"] = ventas["fecha"].dt.to_period("M").dt.to_timestamp()

assert len(ventas) == 481
print("OK 0 —", ventas.shape)

## Grafico 1 — Evolucion mensual

`dt.to_period("M")` agrupa por mes conservando el orden cronologico. Agrupar por
`dt.month_name()` da un eje alfabetico (Abril, Agosto, Diciembre...) — error clasico.

El titulo dice el hallazgo. "Ventas por mes" obliga al lector a sacar la conclusion solo;
para eso no hacia falta el grafico.

In [ ]:
serie = ventas.groupby("mes")["importe"].sum()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(serie.index, serie.values, marker="o", linewidth=2)
ax.fill_between(serie.index, serie.values, alpha=0.12)

mejor, peor = serie.idxmax(), serie.idxmin()
ax.annotate(f"max {serie.max():,.0f}", (mejor, serie.max()),
            textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)

ax.set_title(f"La facturacion oscila entre S/ {serie.min():,.0f} y S/ {serie.max():,.0f} "
             f"al mes, sin tendencia clara", fontsize=11, weight="bold")
ax.set_xlabel("Mes 2025")
ax.set_ylabel("Importe (S/)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
fig.autofmt_xdate(rotation=45)
plt.tight_layout()
plt.show()

print(serie.round(2).to_string())

**Conclusion 1 (referencia):** la serie oscila sin tendencia sostenida — no hay
crecimiento que defender ni caida que apagar. El pico y el valle son ruido de volumen
mensual, no estacionalidad: con un solo ano de datos no se puede afirmar estacionalidad.
Aceptar cualquier lectura del alumno que **no** invente una tendencia que los datos no muestran.

## Grafico 2 — Ranking de productos

Barras horizontales y ordenadas. El ojo compara longitudes bien y angulos mal:
por eso barras y no pastel.

In [ ]:
por_prod = ventas.groupby("id_producto")["importe"].sum().sort_values()
colores = ["#c8102e" if i >= len(por_prod) - 3 else "#4c72b0" for i in range(len(por_prod))]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(por_prod.index, por_prod.values, color=colores)

top3 = por_prod.tail(3).sum() / por_prod.sum() * 100
ax.set_title(f"El top 3 de productos concentra el {top3:.0f}% de la facturacion",
             fontsize=11, weight="bold")
ax.set_xlabel("Importe total (S/)")
ax.set_ylabel("Producto")
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
for i, v in enumerate(por_prod.values):
    ax.text(v, i, f" {v:,.0f}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

**Conclusion 2 (referencia):** la facturacion esta concentrada en pocos productos.
Consecuencia operativa: el quiebre de stock de uno solo de ellos golpea el mes entero;
el presupuesto de marketing deberia seguir esa concentracion, no repartirse parejo.

## Grafico 3 — Canal por ciudad

`pivot_table` + `heatmap`. `annot=True` porque un heatmap sin numeros obliga a estimar
por color, y nadie estima bien por color.

In [ ]:
tabla = ventas.pivot_table(index="ciudad", columns="canal",
                           values="importe", aggfunc="sum", fill_value=0)

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.heatmap(tabla, annot=True, fmt=",.0f", cmap="Blues", linewidths=.5,
            cbar_kws={"label": "Importe (S/)"}, ax=ax)
ax.set_title("El mix de canales no es igual en todas las ciudades",
             fontsize=11, weight="bold")
ax.set_xlabel("Canal")
ax.set_ylabel("Ciudad")
plt.tight_layout()
plt.show()

print((tabla.div(tabla.sum(axis=1), axis=0) * 100).round(1).to_string())

**Conclusion 3 (referencia):** el reparto porcentual por ciudad no es uniforme, asi que
una campana nacional de canal unico deja plata sobre la mesa en las plazas cuyo canal
dominante es otro. La tabla de porcentajes de abajo es la que sustenta la afirmacion,
no el heatmap de valores absolutos — que solo refleja el tamano de cada ciudad.

## Grafico 4 — Distribucion del ticket

Escala logaritmica porque los tickets van de S/ 65 a S/ 19.200. En escala lineal todas
las cajas quedan aplastadas contra el cero y el grafico no dice nada.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=ventas, x="canal", y="importe", ax=ax)
sns.stripplot(data=ventas, x="canal", y="importe", size=2.5,
              color=".25", alpha=.35, ax=ax)
ax.set_yscale("log")
ax.set_title("Muchos tickets chicos y una cola larga de ventas grandes en los 3 canales",
             fontsize=11, weight="bold")
ax.set_xlabel("Canal")
ax.set_ylabel("Importe (S/, escala log)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.tight_layout()
plt.show()

print(ventas.groupby("canal")["importe"].describe().round(0).to_string())

**Conclusion 4 (referencia):** la mediana es baja y la cola es larga en los tres canales.
Por eso **el promedio no sirve como KPI aqui**: una sola laptop mueve la media del mes.
El KPI correcto es la mediana, o el promedio segmentado por rango de ticket.

## 5 — Figura resumen

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1
ax = axes[0, 0]
ax.plot(serie.index, serie.values, marker="o", linewidth=2)
ax.fill_between(serie.index, serie.values, alpha=0.12)
ax.set_title("Facturacion mensual 2025", weight="bold")
ax.set_ylabel("S/")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
ax.tick_params(axis="x", rotation=45)

# 2
ax = axes[0, 1]
ax.barh(por_prod.index, por_prod.values, color=colores)
ax.set_title(f"Top 3 = {top3:.0f}% de la facturacion", weight="bold")
ax.set_xlabel("S/")
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))

# 3
sns.heatmap(tabla, annot=True, fmt=",.0f", cmap="Blues", linewidths=.5,
            cbar=False, ax=axes[1, 0])
axes[1, 0].set_title("Importe por ciudad y canal", weight="bold")

# 4
ax = axes[1, 1]
sns.boxplot(data=ventas, x="canal", y="importe", ax=ax)
ax.set_yscale("log")
ax.set_title("Distribucion del ticket por canal", weight="bold")
ax.set_ylabel("S/ (log)")

fig.suptitle("Ventas 2025 — resumen ejecutivo", fontsize=14, weight="bold")
plt.tight_layout()
fig.savefig("dashboard_ventas.png", dpi=150, bbox_inches="tight")
plt.show()

import os
assert os.path.exists("dashboard_ventas.png")
print("OK 5 —", round(os.path.getsize("dashboard_ventas.png") / 1024), "KB")

## 6 — Recomendacion (referencia)

1. **Proteger el stock del top 3 de productos**, que concentra el {top3}% de la facturacion:
   un quiebre ahi cuesta mas que cualquier campana.
2. **Presupuesto de marketing por plaza, no nacional**: el mix de canales difiere por ciudad
   (ver tabla de porcentajes del grafico 3).
3. **Cambiar el KPI de ticket promedio por ticket mediano.** La cola larga del grafico 4
   hace que una venta grande distorsione el promedio del mes entero.

Se acepta cualquier recomendacion distinta **siempre que cite una cifra de sus propios graficos**.
Lo que no se acepta es "hay que vender mas".

---
### Errores tipicos al corregir

- **Agrupar por `dt.month_name()`**: eje alfabetico, Abril antes que Enero.
- **Titulos que repiten el nombre de la variable** ("Importe por ciudad") en vez de decir el hallazgo.
- **Pastel con 8 categorias.** Ilegible; ademas oculta el orden.
- **Boxplot en escala lineal** con esta distribucion: no se ve nada y el alumno concluye "no hay diferencia".
- **Ejes sin formato de miles**: `1.2e6` no se pega en un correo a gerencia.
- **Graficar sin limpiar**: si no filtro `cantidad > 0`, la devolucion mete un importe negativo en las barras.
- **Conclusiones vacias o genericas.** Es el criterio que mas puntua y el que mas se deja en blanco.
- **`plt.savefig` sin `bbox_inches="tight"`**: el titulo o las etiquetas salen cortadas.